# 09-зертханалық жұмыс. Ойыншылар сипаттамаларының тарихи жиынына EDA жүргізу

**Пән:** Деректерді интеллектуалды талдау  
**Ұзақтығы:** 2 академиялық сағат  
**Алдын ала дайындық:** 6-9 дәрістер; тазалау, groupby, визуализация және корреляция.

## Оқу мақсаттары

- Деректер сөздігінен жаңа гипотезаларға дейінгі есеп жасау.
- Ауытқыма мәндер мен байланыстарды бірнеше кесіндіде тексеру.
- Есептелген фактілер мен жорамалдарды ажырату.

## Ноутбукты іске қосу

Python ортасында мына пакеттерді орнатыңыз: `matplotlib`, `numpy`, `pandas`. Jupyter бағдарламасында Python 3 ядросын таңдап, ұяшықтарды жоғарыдан төмен орындаңыз. Соңғы код ұяшығында тапсырмаларды өз бетіңізше орындаңыз. Нәтижелерді зертханалық жұмыста көрсетілген `work_results` жолына сақтаңыз.

Оқулықтың архивін ашып, `notebooks/` және `data/` қалталарын бірге сақтаңыз. Ноутбукты бөлек жүктесеңіз, Jupyter немесе Colab файлдар панелінде ағымдағы жұмыс қалтасында `data` ішкі қалтасын жасап, оқулықтан алынған `football.csv` файлын сонда орналастырыңыз. Бірінші ұяшық дерек жолын тексереді.


In [ ]:
from pathlib import Path
import os

# Жұмыс қалтасын анықтау.
project_dir = Path.cwd()
if project_dir.name == "notebooks" and (
    (project_dir.parent / "data" / "football.csv").is_file()
    or (project_dir.parent / "index.html").is_file()
):
    project_dir = project_dir.parent

if not (project_dir / "data" / "football.csv").is_file():
    raise FileNotFoundError(
        "data/football.csv табылмады. Оқулықтан football.csv файлын жүктеп алып, "
        "ағымдағы жұмыс қалтасындағы data ішкі қалтасына орналастырыңыз."
    )

if Path.cwd() != project_dir:
    os.chdir(project_dir)

print("Жұмыс қалтасы:", project_dir)


## Аналитикалық есеп және қолжетімсіз деректерді алмастыру

«Тарихи футбол жиынында жас пен дағды үлестірімдері қандай және олар позициялар арасында қалай ерекшеленеді?» деген сұрақты зерттеңіз. Байланыссыз графиктер жинағы емес, қайта орындалатын есеп қажет: паспорт, сапа, бірөлшемді сипаттамалар, топтар, байланыстар, шектеулер және келесі сұрақтар. Барлық есептеу жинақтағы data/football.csv бойынша орындалады.

Бастапқы 9-зертханалық жұмыс cacao және какао шыққан елдердің жазылу нұсқаларын түзетуге арналған. Бұл файл жоқ. Мұндағы тапсырмалар санат жиіліктері, қалыпқа келтіру, IQR, үлестірім визуализациясы және сандық өрістер байланысы дағдыларын сақтап, football жиынына бейімделген. Бастапқы слайдтардағы шоколад сандарын есепке көшіруге болмайды. Санаттық тазалауды жеке жаттықтыру үшін төменде толық берілген синтетикалық Series бар; ол football кестесіне қосылмайды.

Жиында 12897 жол және жетіспейтін ұяшықтары жоқ 42 өріс бар. Бұл — ойыншылардың қазіргі бабына қатысты мәлімет емес, тарихи кескін. Name бірегей ID болуына кепілдік бермейді. Unnamed: 0 — ескі индекс; оны дағдылар сипаттамасы мен корреляциялардан шығарыңыз. Penalties — гол саны емес, пенальти дағдысының рейтингі. Value/Wage валютасы мен кезеңі расталмаған; қаржылық өрістерді осы зерттеуден тыс қалдыруға болады.

Python, pandas, NumPy және Matplotlib қажет. Интернет пен сыртқы жиындар керек емес. Статистикалық тесттер міндетті емес: мұнда қорытындылау мен алғышарттарды тексеру тақырыбынан кейін зерттелетін гипотезалар тұжырымдалады. Тиісті рәсімсіз екі boxplot айырмасын «статистикалық маңызды» деп жарияламаңыз.

## Дайындық және талданған мысал



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

football = pd.read_csv(Path('data') / 'football.csv')
out = Path('work_results') / 'lab09'
out.mkdir(parents=True, exist_ok=True)



Тәуелсіз мысал ретінде ағын шығынының өлшеулерін қарастырамыз. Ол сандық ережені қатенің дәлелімен шатастырмауға үйретеді.



In [ ]:
flow = pd.Series([10., 11., 12., 13., 40.])
q1, q3 = flow.quantile([0.25, 0.75])
iqr = q3 - q1
limits = (q1 - 1.5 * iqr, q3 + 1.5 * iqr)
flags = ~flow.between(*limits)
print(limits, flow.loc[flags].tolist())



Шектер 8 және 16-ға тең, 40 саны белгі алады. Аспап пен жұмыс режимі туралы мәліметсіз оның ақауды немесе нақты жоғары шығынды білдіретіні белгісіз. Тапсырмаларда бастапқы нұсқаны сақтап, үміткерлерді бөлек белгілеңіз. Осы жұмыс шарттары тек гистограмманы тегістеу үшін жолдарды жоюға тыйым салады.

Мәтіндік жолдар жаттығуы үшін тәуелсіз оқу санаттарын дайындаңыз. Бұлар бастапқы CSV елдерін түзету нәтижелері емес, жасанды жазылу нұсқалары.



In [ ]:
origin_demo = pd.Series(
    [' Argentina ', 'argentina', 'ARG', 'Brazil',
     ' brazil ', '', None, 'unknown'],
    dtype='string',
)



## Міндетті тапсырмалар

1. **Паспорт және бақылау бірлігі.** Пішінді, баған тізімін, типтерді және жетіспейтін мәндер санын сақтаңыз. Кемінде сегіз өріске dictionary.csv құрыңыз: column, meaning, semantic_type, unit, limitation. Age, Position, Nationality, Stamina, Acceleration, SprintSpeed, Name және Unnamed: 0 енгізіңіз. Дереккөз туралы белгісіз мәліметті файлдың анықталған сипаттамаларынан ажыратыңыз. Күндерсіз нақты зерттеуге болатын негізгі сұрақ пен екі қосымша сұрақты тұжырымдаңыз.

2. **Сапа аудиті.** Ескі индексті шығарғаннан кейін мазмұнды өрістердегі дәл қайталануларды, қайталанатын Name мәндерін және Position, Club, Nationality бірегей санаттар санын есептеңіз. Бұлар неге әртүрлі тексерулер екенін түсіндіріңіз. quality.csv құрыңыз: check, n_affected, interpretation, action. Мәселе табылмаса, тексеруді өткізіп жібермей, нөлдік нәтижені жазыңыз. Есепті мазмұндырақ ету үшін football ішіне жасанды жетіспейтін мәндер қоспаңыз.

3. **Санаттар және өз бетімен қалыпқа келтіру.** origin_demo түпнұсқасын сақтап, қалыпқа келтірілген кілт құрыңыз: strip, casefold, дәл сәйкестік arg -> argentina, бос жол және unknown -> жетіспейтін мән. NA ескерілген бастапқы және кейінгі жиіліктерді көрсетіңіз. Бірде-бір жол жоғалмағанын дәлелдеңіз. Содан кейін нақты Nationality үшін санаттар санын, жиілігі қатаң түрде 10-нан үлкен санаттар санын және осындай санаттарға кіретін жолдар үлесін табыңыз. Санаттар саны мен бақылаулар саны әртүрлі жауаптар екенін атап өтіңіз.

4. **Бірөлшемді сипаттау.** Age, Acceleration, SprintSpeed, Stamina үшін descriptive.csv дайындаңыз: feature, n, min, q1, median, q3, max, mean, std_ddof1. Өлшем бірліктері мен көлемдерін жазып, бір торда төрт гистограмма құрыңыз. Бір белгі үшін mean мен median салыстырып, осы екі сан бойынша асимметрия туралы қандай қорытынды жасауға болатынын және болмайтынын түсіндіріңіз. min <= q1 <= median <= q3 <= max реті орындалатынын тексеріңіз.

5. **Автоматты жоюсыз IQR.** Age және Stamina үшін IQR шектері мен үміткерлер санын есептеп, feature, lower, upper, n_flagged, share_flagged бар flags.csv сақтаңыз. Name және Position көрсетілген бірнеше үміткер жолын шығарыңыз. Үміткер болмаса, бұл дұрыс нәтиже; оны хабарлап, ереженің ықтимал шектеулерін қарастырыңыз. Жеке салыстырмалы кесіндіде үміткерлермен және оларсыз орташа мән мен медиананы есептеңіз. Негізгі жиынды толық қалдырыңыз. Жою туралы шешімге дейін қандай сыртқы ақпарат алу қажет екенін атаңыз.

6. **Санаттық топтар.** Ең жиі алты Position таңдаңыз. Stamina үшін n, mean, median, q1, q3 есептеп, бірдей осьпен boxplot салыңыз. Қасында алты позиция қамтитын толық жиын үлесін хабарлаңыз. Көрсетілгендердің ең үлкен медианасын атап, мәнін келтіріңіз. Қорытындыны көрсетілмеген позицияларға таратпаңыз. Ойыншылардың жалпы әртектілігі неге ақау емес, мазмұндық ерекшелік болуы мүмкін екенін талқылаңыз.

7. **Сандық байланыстар және кесінділер.** 4-тапсырмадағы төрт белгі үшін Пирсон матрицасын және Age пен Stamina scatter диаграммасын құрыңыз. Әртүрлі белгілер арасында корреляция модулі ең үлкен жұпты таңдаңыз; диагональ мен симметриялы қайталануды шығарыңыз. Age/Stamina үшін бүкіл жиын және бөлек ең жиі үш позиция бойынша корреляция есептеңіз. correlations.csv сақтаңыз: group, feature_x, feature_y, n_pairs, pearson_r. Кесінділерде байланыс бағыты мен шамасының реті сәйкес келе ме, сипаттаңыз.

8. **Бақылаудың тұрақтылығы.** Өз бақылауларыңыздың бірін таңдап, оның безендіруге немесе іріктеуге тәуелділігін тексеріңіз. Ықтимал тексерулер: басқа гистограмма аралықтары, жиі алты және он позиция, орташа мән мен медиана, толық жиын және IQR үміткерлері жоқ кесінді. Екі тексеру орындап, сандарды көрсетіңіз. Бақылау өзгерсе, жасырмаңыз: тұжырым мен ықтимал себебін нақтылаңыз. Ыңғайлы нұсқаны таңдамау үшін журналда тексеруді нәтижесін қараудан бұрын атаңыз.

9. **Есеп және келесі гипотезалар.** Кестелері және өз суреттеріңізге сілтемелері бар 600-900 сөз дайындаңыз. Негізделген үш бақылауды, кемінде екі шектеуді және әрі қарай зерттеуге арналған екі гипотезаны қосыңыз. Әр гипотеза үшін салыстырылатын шамаларды, ықтимал араластыруды және жаңа деректерді не тексеру жоспарын атаңыз. Шешімдер тізімімен аяқтаңыз: не сақталды, не белгілерден шығарылды және не дереккөзді нақтылауды талап етеді.

## Не тапсыру және қалай тексеру керек

Есебі бар notebook немесе скрипт, dictionary.csv, quality.csv, descriptive.csv, flags.csv, correlations.csv және топтар кестесін тапсырыңыз. Кемінде төрт сурет: үлестірімдер, boxplot, scatter, корреляция матрицасының кескіні немесе оның оқуға ыңғайлы кестесі. Барлық нәтиже work_results/lab09 ішінде болады. Кестелерді сақтау кездейсоқ индекс қоспауы тиіс; корреляция матрицасының атаулы жолдарын index=True арқылы саналы түрде сақтауға болады.

Есепте «факт - дәлел - шектеу» тізбегін қолданыңыз. Факт есептелген көрсеткішті, дәлел кесте немесе сурет нөмірін, шектеу түсіндірудің нақты шегін қамтуы тиіс. «Әсер бар» сөйлемі байланыс сипаттамасын алмастырмайды. Балама түсіндірме тұжырымдасаңыз, оны гипотеза деп белгілеп, тексеруге не көмектесетінін көрсетіңіз.

Қайта орындау мүмкіндігін бақылау үшін құжатты басынан орындаңыз. Кестелердегі n мәнін сүзгі шарттарымен салыстырыңыз. Жиіліктер қосындысында санаттар жоғалмағанын, үлестер [0, 1] ішінде, шашырау нөл емес болғанда r мәндері [-1, 1] ішінде екенін және квартиль шектері ретімен орналасқанын тексеріңіз. Бастапқы football 12897 жолды сақтағанын және салыстырмалы сүзгілер кезінде өзгермегенін бөлек растаңыз. График пен топтар кестесі позициялардың бірдей ретін қолдануы тиіс.

Қосымша тапсырма. Сол кесінділерде Age және Stamina үшін Спирмен корреляциясын есептеңіз. Пирсонмен салыстырып, айырмашылыққа не себеп болуы мүмкін екенін түсіндіріңіз. Бұл — байланысты сипаттау, сызықтық емес механизмнің автоматты дәлелі емес. p-мәнін тексеруді осы зертханалық жұмыс шегінен тыс қалдырыңыз.

## Қорғау сұрақтары

1. football шығу тегі туралы қандай мәлімет белгілі және қайсысы жоқ?
2. origin_demo неге кез келген ішкі жолмен емес, сөздік бойынша тазаланады?
3. IQR белгісі пәндік тұрғыдан мүмкін емес мәннен қалай ерекшеленеді?
4. Позиция ішіндегі корреляция жалпы корреляциядан неге ерекшеленуі мүмкін?
5. EDA кейінгі гипотезаны оның тәуелсіз расталуынан қалай ажыратуға болады?

## Бағалау рубрикасы және дереккөздер

| Критерий | Балл | Растау |
|---|---|---|
| Паспорт және сөздік | 10 | Бірлік, өрістер, шектеулер |
| Аудит және санаттар | 15 | Жиіліктер, NA, қайталануды тексеру |
| Үлестірімдер | 15 | Төрт белгі және квартильдік бақылау |
| Ауытқыма мәндер және шешімдер | 15 | Негізсіз жоюсыз белгілер |
| Топтар және байланыстар | 20 | Көлемдер, корреляциялар, кесінділер |
| Тұрақтылық | 10 | Сандары бар екі салыстыру |
| Есеп және гипотезалар | 15 | Дәлелдер және қорытынды шектері |
| Барлығы | 100 | |

- [pandas: негізгі есептеулер](https://pandas.pydata.org/docs/user_guide/basics.html)
- [pandas: топтау](https://pandas.pydata.org/docs/user_guide/groupby.html)
- [NIST: EDA кіріспесі](https://www.itl.nist.gov/div898/handbook/eda/section1/eda11.htm)

## Өзіндік жұмыс

Осы зертханалық жұмыстың тапсырмаларын төмендегі ұяшықта орындаңыз. Қажет болса, код пен түсіндірме үшін қосымша ұяшықтар қосыңыз. Есептеулерді аяқтаған соң ядроны қайта іске қосып, ұяшықтарды басынан бастап орындаңыз.

In [ ]:
# Тапсырмаларды орындау кодын осы жерге жазыңыз.
